In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 21:29:26 WARN Utils: Your hostname, Chirags-Laptop.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.117 instead (on interface en0)
26/06/21 21:29:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 21:29:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
df_green = spark.read.parquet('/Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/green/*/*')

26/06/21 21:35:41 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/green/*/*.
java.io.FileNotFoundException: File /Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$ca

In [6]:
df_yellow = spark.read.parquet('/Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/yellow/*/*')

26/06/21 21:34:36 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/yellow/*/*.
java.io.FileNotFoundException: File /Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/code/data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$

In [8]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge']

In [10]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [11]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [12]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [13]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [14]:
from pyspark.sql import functions as F

In [15]:
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [16]:
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [17]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [18]:
df_trips_data.groupBy('service_type').count().show()

[Stage 5:===============================================>         (15 + 3) / 18]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [19]:
df_trips_data.registerTempTable('trips_data')

/Users/chiraggehlot/Desktop/Data Engineering/data-eng-module-6/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [20]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 8:==================================================>      (16 + 2) / 18]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [21]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [23]:
# For visualising the data clearly

df = df_result.limit(10).toPandas()
df

,revenue_zone,revenue_month,service_type,revenue_monthly_fare,revenue_monthly_extra,revenue_monthly_mta_tax,revenue_monthly_tip_amount,revenue_monthly_tolls_amount,revenue_monthly_improvement_surcharge,revenue_monthly_total_amount,revenue_monthly_congestion_surcharge,avg_monthly_passenger_count,avg_monthly_trip_distance
0,112,2020-01-01,green,21761.03,1034.00,703.0,2487.42,308.61,452.7,27470.11,742.25,1.276641,3.478337
1,232,2020-02-01,green,5079.37,476.75,17.0,0.00,184.35,52.8,5810.27,0.00,1.000000,7.241875
2,58,2020-02-01,green,1198.69,80.75,13.0,76.56,36.72,12.6,1425.77,16.50,1.192308,5.657708
3,245,2020-02-01,green,78.71,2.75,1.0,0.00,12.24,0.6,95.30,0.00,1.000000,12.145000
4,234,2020-02-01,green,1156.73,110.00,2.5,0.00,55.08,11.7,1336.01,0.00,1.000000,7.341795
5,117,2020-01-01,green,41693.88,1584.25,122.0,6.29,2525.03,270.9,46202.35,0.00,1.300000,13.908787
6,174,2020-02-01,green,26850.76,1982.50,280.0,365.88,895.38,324.3,30758.32,49.50,1.274354,5.429607
7,210,2020-02-01,green,36606.00,2514.25,692.5,1437.66,667.59,558.6,42487.60,33.00,1.175875,4.499913
8,85,2020-02-01,green,14506.02,1340.50,123.0,96.23,230.91,174.3,16511.66,16.50,1.196532,4.288293
9,192,2020-01-01,green,8618.38,395.25,63.5,374.93,54.05,90.3,9608.11,0.00,1.306011,5.159457


In [25]:
# Reduce the number of partitions of files in one parquet file 

df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')